# Pipeline test for the KuKi LLM experiments

Runs every script end-to-end on **synthetic Label Studio exports** with **mock models**
(random but schema-valid answers). No GPU needed. Everything is written to `test_run/`.

Numbers produced here are meaningless; the test checks that files, joins and metrics work.
Run the cells top to bottom in a fresh kernel.

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
import json
import os
import shutil
from pathlib import Path

# Redirect all data and prompt paths BEFORE any project module is imported.
TEST_DIR = Path("test_run")
shutil.rmtree(TEST_DIR, ignore_errors=True)
os.environ["KUKI_DATA_DIR"] = str(TEST_DIR / "data")
os.environ["KUKI_PROMPT_DIR"] = str(TEST_DIR / "prompts")

# Dummy codebook prompts; the real wrappers.json is reused.
prompt_dir = TEST_DIR / "prompts"
prompt_dir.mkdir(parents=True)
shutil.copy("prompts/wrappers.json", prompt_dir)
for layer in ["L1", "L2", "L3", "L4"]:
    for lang in ["en", "ru", "tr"]:
        (prompt_dir / f"codebook_{layer}_{lang}.md").write_text(f"Test codebook {layer} ({lang})", encoding="utf-8")

## 1. Synthetic exports
Eight Russian articles, three annotators (one export per annotator, like the real projects).
Coverage mimics the block design: triple, double, single and not annotated.

In [2]:
def make_content(i):
    return (f"Газ и безопасность {i}\n\n"
            "Москва угрожает Европе прекращением поставок газа.\n\n"
            "Жители Киева пострадали от обстрелов.\n\n"
            "Эксперты предупреждают о кризисе.")


def span(content, text, control, label):
    start = content.index(text)
    return {"id": f"{control}-{start}", "from_name": control, "to_name": "content", "type": "labels",
            "value": {"start": start, "end": start + len(text), "text": text, "labels": [label]}}


def frames(*names):
    return {"id": "frames", "from_name": "l2_frames", "to_name": "content", "type": "choices",
            "value": {"choices": list(names)}}


def coded(content, text, literal, insider, why):
    region = span(content, text, "l4_coded", "Coded Phrase")
    fields = [{"id": region["id"], "from_name": name, "to_name": "content", "type": "textarea",
               "value": {"start": region["value"]["start"], "end": region["value"]["end"], "text": [value]}}
              for name, value in [("l4_literal", literal), ("l4_insider", insider), ("l4_why", why)]]
    return [region] + fields


def annotation(annotator, content, i):
    # Even articles: all annotators agree on 'Economic' and a 'Call' span -> human alpha is defined.
    shared = [frames("Economic")] if i % 2 == 0 else []
    shared += [span(content, "о кризисе", "l3_persuasion", "Call")] if i % 2 == 0 else []
    return shared + own_annotation(annotator, content)


def own_annotation(annotator, content):
    if annotator == "A":
        return [frames("Security & defense", "Political"),
                span(content, "Москва", "l1_roles", "ANTAGONIST"),
                span(content, "Европе", "l1_roles", "INNOCENT"),
                span(content, "угрожает Европе", "l3_persuasion", "Manipulative Wording"),
                *coded(content, "прекращением поставок газа", "stopping gas", "energy as weapon", "context")]
    if annotator == "B":
        return [frames("Security & defense"),
                span(content, "Москва", "l1_roles", "ANTAGONIST"),
                span(content, "Жители Киева", "l1_roles", "INNOCENT"),
                span(content, "Эксперты предупреждают", "l3_persuasion", "Justification")]
    return [frames("Security & defense", "External regulation & reputation"),
            span(content, "Европе", "l1_roles", "INNOCENT"),
            span(content, "пострадали от обстрелов", "l3_persuasion", "Manipulative Wording")]


COVERAGE = ["ABC", "ABC", "AB", "BC", "A", "B", "C", ""]
SOURCES = ["ria_novosti", "theinsider"]

raw_dir = TEST_DIR / "data" / "raw"
raw_dir.mkdir(parents=True)
for letter in "ABC":
    tasks = []
    for i, covered_by in enumerate(COVERAGE):
        content = make_content(i)
        annotations = [{"id": i, "result": annotation(letter, content, i), "was_cancelled": False}] if letter in covered_by else []
        tasks.append({"id": 100 + i, "annotations": annotations,
                      "data": {"content": content, "source": SOURCES[i % 2], "author": f"Автор {i % 3}"}})
    (raw_dir / f"ru_{letter}.json").write_text(json.dumps(tasks, ensure_ascii=False), encoding="utf-8")
sorted(p.name for p in raw_dir.iterdir())

['ru_A.json', 'ru_B.json', 'ru_C.json']

## 2. Structural check of an export
Run this on a **real** export first (e.g. `Path("data/raw/ru_A.json")`) to confirm field
names before trusting `prep_00`: task data keys, control names, annotation counts.

In [3]:
from collections import Counter


def inspect_export(path):
    tasks = json.loads(path.read_text(encoding="utf-8"))
    print(f"{path.name}: {len(tasks)} tasks")
    print("task data keys:", sorted(tasks[0]["data"].keys()))
    print("annotations per task:", Counter(len(t.get("annotations", [])) for t in tasks))
    controls = Counter(r["from_name"] for t in tasks for a in t.get("annotations", []) for r in a["result"])
    print("controls:", dict(controls))


inspect_export(raw_dir / "ru_A.json")

ru_A.json: 8 tasks
task data keys: ['author', 'content', 'source']
annotations per task: Counter({1: 4, 0: 4})
controls: {'l2_frames': 7, 'l3_persuasion': 7, 'l1_roles': 8, 'l4_coded': 4, 'l4_literal': 4, 'l4_insider': 4, 'l4_why': 4}


## 3. Data preparation and splits

In [4]:
%run prep_00_parse_exports.py --langs ru

/Users/christophhau/Projekte/KuKi_experiments/venvKuki/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


articles per language x number of annotators:
n_annotators  0  1  2  3
lang                    
ru            1  3  2  2
human label rows: 599, L4 spans: 4


In [5]:
import pandas as pd
import config
from llm_io import read_jsonl

articles = read_jsonl(config.PREP_DIR / "articles.jsonl")
print("entities of article 0:", articles[0]["entities"])
labels = pd.read_csv(config.PREP_DIR / "human_labels.csv")
# Expected: Москва/Европе/Жители Киева as entities; frames and paragraph labels per annotator.
labels[labels["value"] == 1].head(12)

entities of article 0: [{'key': 'европа', 'display': 'Европе'}, {'key': 'житель киев', 'display': 'Жители Киева'}, {'key': 'москва', 'display': 'Москва'}]


,lang,article_id,annotator,layer,unit,label,value
0,ru,656e904cbe74,ru_A,L2,doc,Economic,1
7,ru,656e904cbe74,ru_A,L2,doc,Security & defense,1
12,ru,656e904cbe74,ru_A,L2,doc,Political,1
25,ru,656e904cbe74,ru_A,L3,p1,Manipulative Wording,1
36,ru,656e904cbe74,ru_A,L3,p3,Call,1
40,ru,656e904cbe74,ru_A,L1,e:европа,INNOCENT,1
45,ru,656e904cbe74,ru_A,L1,e:москва,ANTAGONIST,1
54,ru,6022ab5065ac,ru_A,L2,doc,Security & defense,1
59,ru,6022ab5065ac,ru_A,L2,doc,Political,1
72,ru,6022ab5065ac,ru_A,L3,p1,Manipulative Wording,1


In [6]:
%run prep_01_make_splits.py --n-dev 1 --n-l4 2

not annotated (excluded everywhere): 1
dev                  {'ru': 1}
s1_main_grid         {'ru': 4}
s2_stability         {'ru': 4}
s3_metadata_probe    {'ru': 4}
s4_generalisation    {'ru': 2}
s5_l4_pilot          {'ru': 2}


## 4. Prompt check
What the model actually sees: one paragraph-with-context call, native prompt language.

In [7]:
import llm_io

article = llm_io.load_split("s1_main_grid")[0]
for message in llm_io.build_messages(article, "L3", "para_ctx", "native", target=2):
    print(f"--- {message['role']} ---\n{message['content']}\n")
print(json.dumps(llm_io.output_schema("L1", llm_io.entity_names(article)), ensure_ascii=False)[:300])

--- system ---
Test codebook L3 (ru)

--- user ---
Полный текст статьи, только для контекста (абзацы пронумерованы [P0], [P1], [P2] и т. д.):

[P0] Газ и безопасность 0

[P1] Москва угрожает Европе прекращением поставок газа.

[P2] Жители Киева пострадали от обстрелов.

[P3] Эксперты предупреждают о кризисе.

Используя статью выше как контекст, разметьте только абзац [P2] в соответствии с кодбуком:

[P2] Жители Киева пострадали от обстрелов.

Отвечайте только в формате JSON в требуемой структуре.

{"type": "object", "required": ["roles"], "properties": {"roles": {"type": "array", "items": {"type": "object", "required": ["entity", "role"], "properties": {"entity": {"type": "string", "enum": ["Европе", "Жители Киева", "Москва"]}, "role": {"type": "string", "enum": ["PROTAGONIST", "ANTAGONIST", 


## 5. Stage 1: main grid (mock models)
Two mock models give two sizes and two profiles, so the decomposition has something to split.

In [8]:
%run s1_main_grid.py --model mock-M --dev
%run s1_main_grid.py --model mock-M
%run s1_main_grid.py --model mock-S

s1_dev__mock-M.jsonl: 54 jobs, 0 already done, 54 to run
s1_main_grid__mock-M.jsonl: 216 jobs, 0 already done, 216 to run
  100/216
  200/216
s1_main_grid__mock-S.jsonl: 216 jobs, 0 already done, 216 to run
  100/216
  200/216


In [9]:
# Re-running must resume: expect '0 to run'.
%run s1_main_grid.py --model mock-S

s1_main_grid__mock-S.jsonl: 216 jobs, 216 already done, 0 to run


In [10]:
# One prediction file per experiment and model.
print(sorted(p.name for p in config.PRED_DIR.iterdir()))
predictions = pd.DataFrame(read_jsonl(config.PRED_DIR / "s1_main_grid__mock-M.jsonl") + read_jsonl(config.PRED_DIR / "s1_main_grid__mock-S.jsonl"))
print(predictions.groupby(["model", "layer", "granularity", "prompt_lang"]).size().unstack())
predictions[["key", "raw", "parse_ok"]].head(3)

['s1_dev__mock-M.jsonl', 's1_main_grid__mock-M.jsonl', 's1_main_grid__mock-S.jsonl']
prompt_lang               en  native
model  layer granularity            
mock-M L1    doc           4       4
             para         16      16
             para_ctx     16      16
       L2    doc           4       4
             para         16      16
             para_ctx     16      16
       L3    doc           4       4
             para         16      16
             para_ctx     16      16
mock-S L1    doc           4       4
             para         16      16
             para_ctx     16      16
       L2    doc           4       4
             para         16      16
             para_ctx     16      16
       L3    doc           4       4
             para         16      16
             para_ctx     16      16


,key,raw,parse_ok
0,s1_main_grid|mock-M|656e904cbe74|L1|doc|en|non...,"{""roles"": []}",True
1,s1_main_grid|mock-M|656e904cbe74|L1|doc|native...,"{""roles"": []}",True
2,s1_main_grid|mock-M|656e904cbe74|L1|para|en|no...,"{""roles"": []}",True


In [11]:
%run evaluate.py --experiment s1_main_grid

     model lang layer granularity prompt_lang metadata      q  alpha_pair  persp_acc  parse_fail  error_rate
0   mock-M   ru    L1         doc          en     none  0.015      -0.117      0.861         0.0         0.0
1   mock-M   ru    L1         doc      native     none  0.080      -0.100      0.889         0.0         0.0
2   mock-M   ru    L1        para          en     none  0.377      -0.029      0.639         0.0         0.0
3   mock-M   ru    L1        para      native     none  0.059      -0.139      0.639         0.0         0.0
4   mock-M   ru    L1    para_ctx          en     none  0.337      -0.234      0.583         0.0         0.0
5   mock-M   ru    L1    para_ctx      native     none -0.081      -0.131      0.694         0.0         0.0
6   mock-M   ru    L2         doc          en     none  0.304      -0.139      0.857         0.0         0.0
7   mock-M   ru    L2         doc      native     none  0.625      -0.130      0.804         0.0         0.0
8   mock-M   ru    

Expected here: a `SingularMatrixWarning`, because in the mock grid size and profile are confounded (mock-S = S + english, mock-M = M + multilingual). The real grid crosses them.

In [12]:
%run analyze_s1_decomposition.py

lang layer                          term  share  sum_sq  df      F  PR(>F)
  ru    L1                    C(profile)  0.361   0.192 1.0  3.662   0.196
  ru    L1                      Residual  0.197   0.105 2.0    NaN     NaN
  ru    L1 C(granularity):C(prompt_lang)  0.135   0.071 2.0  0.682   0.595
  ru    L1            C(size):C(profile)  0.105   0.056 1.0  1.059   0.412
  ru    L1     C(profile):C(granularity)  0.089   0.047 2.0  0.449   0.690
  ru    L1                       C(size)  0.083   0.044 1.0  0.844   0.455
  ru    L1     C(profile):C(prompt_lang)  0.018   0.009 1.0  0.178   0.714
  ru    L1        C(size):C(prompt_lang)  0.010   0.005 1.0  0.105   0.777
  ru    L1        C(size):C(granularity)  0.002   0.001 2.0  0.011   0.989
  ru    L1                C(prompt_lang) -0.000  -0.000 1.0 -0.000   1.000
  ru    L1                C(granularity) -0.000  -0.000 2.0 -0.000   1.000
  ru    L2                    C(profile)  0.693   0.786 1.0 30.120   0.032
  ru    L2 C(granularity)

/Users/christophhau/Projekte/KuKi_experiments/analyze_s1_decomposition.py:27: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  table = anova_lm(smf.ols(formula, data=g).fit(), typ=2)
/Users/christophhau/Projekte/KuKi_experiments/analyze_s1_decomposition.py:27: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  table = anova_lm(smf.ols(formula, data=g).fit(), typ=2)
/Users/christophhau/Projekte/KuKi_experiments/analyze_s1_decomposition.py:27: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  table = anova_lm(smf.ols(formula, data=g).fit(), typ=2)


## 6. Stages 2-5 (mock models)

In [13]:
%run s2_stability.py --model mock-M --granularity para_ctx --prompt-lang en --n-samples 3
%run evaluate.py --experiment s2_stability

s2_stability__mock-M.jsonl: 144 jobs, 0 already done, 144 to run
  100/144
    model lang layer granularity prompt_lang metadata      q  alpha_pair  persp_acc  parse_fail  error_rate
0  mock-M   ru    L1    para_ctx          en     none -0.029      -0.102      0.667         0.0         0.0
1  mock-M   ru    L2    para_ctx          en     none  0.510      -0.211      0.607         0.0         0.0
2  mock-M   ru    L3    para_ctx          en     none  0.241      -0.026      0.844         0.0         0.0


In [14]:
%run s3_metadata_probe.py --model mock-M
%run evaluate.py --experiment s3_metadata_probe
pd.read_csv(config.RESULT_DIR / "s3_metadata_probe_prevalence_by_meta.csv").head()

s3_metadata_probe__mock-M.jsonl: 36 jobs, 0 already done, 36 to run
    model lang layer granularity prompt_lang metadata      q  alpha_pair  persp_acc  parse_fail  error_rate
0  mock-M   ru    L1         doc          en     none  0.196       0.037      0.944         0.0         0.0
1  mock-M   ru    L1         doc          en  swapped  0.875       0.046      0.944         0.0         0.0
2  mock-M   ru    L1         doc          en     true  0.319       0.032      0.917         0.0         0.0
3  mock-M   ru    L2         doc          en     none  0.750      -0.159      0.875         0.0         0.0
4  mock-M   ru    L2         doc          en  swapped  1.000      -0.127      0.929         0.0         0.0
5  mock-M   ru    L2         doc          en     true  0.010      -0.308      0.804         0.0         0.0
6  mock-M   ru    L3         doc          en     none  0.119      -0.090      0.906         0.0         0.0
7  mock-M   ru    L3         doc          en  swapped  0.346      -0

,model,lang,layer,label,true_pole,none,swapped,true,shift_true_vs_none,shift_swapped_vs_true
0,mock-M,ru,L1,ANTAGONIST,gov_close,0.000000,0.166667,0.000000,0.000000,0.166667
1,mock-M,ru,L1,ANTAGONIST,gov_distant,0.000000,0.000000,0.166667,0.166667,-0.166667
2,mock-M,ru,L1,INNOCENT,gov_close,0.166667,0.000000,0.000000,-0.166667,0.000000
3,mock-M,ru,L1,INNOCENT,gov_distant,0.166667,0.000000,0.500000,0.333333,-0.500000
4,mock-M,ru,L1,PROTAGONIST,gov_close,0.166667,0.166667,0.000000,-0.166667,0.166667


In [15]:
%run s4_generalisation.py --model mock-S --granularity doc --prompt-lang en
%run evaluate.py --experiment s4_generalisation

s4_generalisation__mock-S.jsonl: 6 jobs, 0 already done, 6 to run
    model lang layer granularity prompt_lang metadata   q  alpha_pair  persp_acc  parse_fail  error_rate
0  mock-S   ru    L1         doc          en     none NaN       0.167      0.556         0.0         0.0
1  mock-S   ru    L2         doc          en     none NaN       0.000      0.786         0.0         0.0
2  mock-S   ru    L3         doc          en     none NaN      -0.100      0.875         0.0         0.0


In [16]:
%run s5_l4_pilot.py --model mock-M
%run evaluate.py --experiment s5_l4_pilot

s5_l4_pilot__mock-M.jsonl: 4 jobs, 0 already done, 4 to run
    model lang layer granularity prompt_lang metadata  recall  precision  n_human_paragraphs  n_flagged_paragraphs
0  mock-M   ru    L4         doc          en     none     0.0       0.00                   1                     4
1  mock-M   ru    L4         doc      native     none     1.0       0.25                   1                     4


## 7. Smoke test with a real model
Use a **fresh kernel without the test cells above**, after `prep_00`/`prep_01` ran on the real
data, on the GPU machine. Loads the model with transformers and runs one constrained call:
check that the output is valid JSON, and look at token count and time per call.

In [ ]:
from tqdm.auto import tqdm

RUN_REAL = True
print("RUN_REAL is set to", RUN_REAL)
if RUN_REAL:
    import llm_io
    print("Loading model and running a smoke test on the first article of the dev split...")
    backend = llm_io.load_model("olmo3-7b")
    print("Model loaded. Running smoke test...")
    article = llm_io.load_split("dev")[0]
    jobs = list(llm_io.make_jobs("smoke", [article], "olmo3-7b", ["L2", "L3"], ["doc"], ["en"]))
    for job in tqdm(jobs, desc="Smoke test"):
        record = llm_io.call_llm(backend, job)
        tqdm.write(f"{job['layer']} {record['parsed']} | tokens in/out: {record['tokens_in']} {record['tokens_out']} "
                   f"| seconds: {record['seconds']} | error: {record['error']}")

RUN_REAL is set to True
Loading model and running a smoke test on the first article of the dev split...


Fetching 3 files:   0%|          | 0/3 [02:58<?, ?it/s]


In [1]:
import time, torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
print(transformers.__version__, torch.cuda.is_available())

hf = "allenai/Olmo-3-7B-Instruct"
t = time.time(); tok = AutoTokenizer.from_pretrained(hf); print("tokenizer", time.time() - t)
t = time.time(); llm = AutoModelForCausalLM.from_pretrained(hf, dtype="auto", device_map="auto"); print("model", time.time() - t)
t = time.time(); data = build_token_enforcer_tokenizer_data(tok); print("enforcer", time.time() - t)

/Users/christophhau/Projekte/KuKi_experiments/venvKuki/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4.57.6 False
tokenizer 0.7336008548736572


Fetching 3 files:   0%|          | 0/3 [03:18<?, ?it/s]
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 